In [3]:
import os
import sys
import socket
import xarray as xr
import xarray_regrid
import numpy as np
import importlib
from concurrent.futures import ProcessPoolExecutor
from functools import partial
import re
from datetime import datetime, timedelta, date
from pathlib import Path
import xesmf as xe


ModuleNotFoundError: No module named 'xesmf'

In [5]:
def get_python_path():
    hostname = socket.gethostname()                                 # 1. Identify the computer by hostname
    code_locations = {                                              # 2. Set default Python code location based on hostname
        "NECMAC04363461.local": "/Users/kimberly.hyde/Documents/",  # Mac laptop
        "nefscsatdata": "/mnt/EDAB_Archive/",                       # Satdata
        "guihyde": "/mnt/EDAB_Archive/"                             # Kim's Satdata container
    }

    default_code_path = os.path.join(code_locations.get(hostname),"nadata/python/utilities/py")    
    if not os.path.isdir(default_code_path):
        print(f"Directory not found: {default_code_path}")
        return {}
    print(f"Default untilities path: {default_code_path}")
    return default_code_path

def global_import(function_map,verbose=False):
    for module_name, function_names in function_map.items():
        try:
            module = importlib.import_module(module_name)
        except ModuleNotFoundError:
            print(f"❌ Module '{module_name}' could not be imported.")
            continue

        for name in function_names:
            try:
                func = getattr(module, name)
                globals()[name] = func
                print(f"✅ Imported: {name} from {module_name}")
            except AttributeError:
                print(f"⚠ Function '{name}' not found in '{module_name}'")
    

# Get the path to the utilities folder
util_path = get_python_path()

# Add to sys.path
if util_path not in sys.path:
    sys.path.append(util_path)

# Find the default utiltity functions and import into global
from import_utilities import get_pyfile_functions
functions = get_pyfile_functions()
gl = global_import(functions)

# Import local calc functions
calc_functions = get_pyfile_functions(os.getcwd(),"calc")
gl = global_import(calc_functions)



Default untilities path: /Users/kimberly.hyde/Documents/nadata/python/utilities/py
✅ Imported: get_python_dir from import_utilities
✅ Imported: get_pyfile_functions from import_utilities
✅ Imported: import_utility_functions from import_utilities
✅ Imported: dataset_defaults from getfiles
✅ Imported: product_defaults from getfiles
✅ Imported: get_datasets_source from getfiles
✅ Imported: get_dataset_dirs from getfiles
✅ Imported: get_dataset_products from getfiles
✅ Imported: get_prod_files from getfiles
✅ Imported: pixel_area from calc_mapped_pixel_area
✅ Imported: total_annual_pp from calc_annual_pp
✅ Imported: day_length_calculation from calc_day_length
✅ Imported: vgpm_models from calc_primary_production
✅ Imported: daily_vgpm from calc_primary_production


In [ ]:
def file_make(input_files, output_file):
    
    output_mtime = Path(output_file).stat().st_mtime
    newer_found = False
    for file in input_files:
        if os.path.getmtime(file) > output_mtime:
            print(f"↪ {file} newer than output file: {output_file}, recreate")
            newer_found = True
    return newer_found

def daily_vgpm(date, chl, sst, par, lat, lon, output_dir,input_dirs):
    #d8 = str(date).split('T')[0]
    
    # Define the output file and check if exists and is up to date
    output_file = os.path.join(output_dir, f"D_{date}-OCCCI-V6.0-GLOBAL_MAPPED-PPD.nc")
    should_process = True
    
    
    if os.path.exists(output_file):
        if not file_make(input_files, output_file)
    #    if not any_input_newer(input_dirs, date, output_file):
    #        expected = chl.sel(time=str(date)).time.dt.floor("D").values
    #    if not missing_dates(output_file, expected):
        print(f"✓ Skipping {date} — output file exists.")
        return
    #        should_process = False
    #if not should_process:

    print(f"The output file is: {output_file}")
    
    # Subset dataset to the specified year
    chl_day = chl.sel(time=str(date))
    sst_day = sst.sel(time=str(date))
    par_day = par.sel(time=str(date))

    # Broadcast DOY and LAT for day length calculation
    doy = chl_day['time'].dt.dayofyear
    doy_3d, lat_3d = xr.broadcast(doy, lat)

    # Calculate the day length
    daylength = day_length_calculation(lat_3d.values, doy_3d.values)
    
    # Convert day_length to 2D array
    daylength = np.tile(daylength.reshape(-1, 1), (1, len(lon)))
    
    # Convert day_length to dataset
    day_length = xr.DataArray(
        data=daylength,
        dims=chl_day.dims,
        coords=chl_day.coords,
        name="day_length"
    )
    
    # Change variables to float32
    chl_day = chl_day.astype("float32")
    sst_day = sst_day.astype("float32")
    par_day = par_day.astype("float32")
    day_length = day_length.astype("float32")
   
    print("Running vgpm_models...")
    pp_eppley, pp_vgpm, kdpar, chl_eu, zeu =vgpm_models(chl_day,sst_day,par_day,day_length)
    
    
    #Create data arrays for each variable
    xa_pp_eppley = xr.DataArray(pp_eppley,dims=chl_day.dims,coords=chl_day.coords,name="pp_eppley")
    xa_pp_vgpm = xr.DataArray(pp_vgpm,dims=chl_day.dims,coords=chl_day.coords,name="pp_vgpm")
    xa_kdpar = xr.DataArray(kdpar,dims=chl_day.dims,coords=chl_day.coords,name="pp_kdpar")
    xa_chl_euphotic = xr.DataArray(chl_eu,dims=chl_day.dims,coords=chl_day.coords,name="chl_euphotic")
    xa_zeu = xr.DataArray(zeu,dims=chl_day.dims,coords=chl_day.coords,name="zeu")
    
    
    # Create a dataset
    pp_dataset = xr.Dataset({
        "pp_eppley": xa_pp_eppley,
        "pp_vgpm": xa_pp_vgpm,
        "kdpar": xa_kdpar,
        "chl_euphotic": xa_chl_euphotic,
        "zeu": xa_zeu
    })
    

    pp_dataset = pp_dataset.expand_dims({"time":[date]})
    
    print(f"✓ Writing {output_file}")
    pp_dataset.to_netcdf(output_file)

In [ ]:
def load_daily_inputs(date, chl_path, sst_path, par_path):
    chl = xr.open_dataset(chl_path, chunks={"lat": 100, "lon": 100})
    sst = xr.open_dataset(sst_path, chunks={"lat": 100, "lon": 100})
    par = xr.open_dataset(par_path, chunks={"lat": 100, "lon": 100})
    return chl, sst, par


def process_one_date(date, paths, lat, lon, output_dir, input_dirs):
    chl_path, sst_path, par_path = paths
    chl, sst, par = load_daily_inputs(date, chl_path, sst_path, par_path)
    daily_vgpm(date, chl, sst, par, lat, lon, output_dir, input_dirs)


def process_one_date(date, paths, output_dir, input_dirs):
    """
    Processes a single day's CHL, SST, PAR maps to compute VGPM globally.

    Parameters:
        date (str): Date in "YYYY-MM-DD" format.
        paths (tuple): (chl_path, sst_path, par_path)
        output_dir (str): Directory to save results.
        input_dirs (dict): Optional extra data sources, e.g. ancillary files.

    Returns:
        None
    """

    chl_path, sst_path, par_path = paths

    out_file = os.path.join(output_dir, f"D_{date}-OCCCI-V6.0-GLOBAL_MAPPED-PPD.nc")
    if os.path.exists(out_file):
        if not file_make(paths, out_file)
        print(f"✓ Skipping {date} — output file exists.")
        return
        

    # Check if out_file exists and if so, if the input data are newer than the out_file

    try:
        chl_ds = xr.open_dataset(chl_path)
        sst_ds = xr.open_dataset(sst_path)
        par_ds = xr.open_dataset(par_path)

        chl = chl_ds["chlor_a"]
        sst = sst_ds["analysed_sst"]
        par = par_ds["PAR_mean"]

        # Regrid using xarray-regrid
        if verbose:
            print("Regridding...")
        sst = sst.regrid.linear(chl_ds)
        par = par.regrid.linear(chl_ds)

        # Assuming lat/lon grids are identical and present in CHL file
        lat = chl_ds["lat"].values
        lon = chl_ds["lon"].values

        # Mask valid pixels
        valid_mask = ~np.isnan(chl) & ~np.isnan(sst) & ~np.isnan(par)

        result_array = np.full(chl.shape, np.nan)

        # Loop through valid pixels
        for i, j in zip(*np.where(valid_mask)):
            chl_val = chl[i, j]
            sst_val = sst[i, j]
            par_val = par[i, j]
            lat_val = lat[i]
            lon_val = lon[j]

            result_array[i, j] = vgpm_models(chl_val, sst_val, par_val, lat_val, lon_val, date)

        # Save result
        out_file = os.path.join(output_dir, f"D_{date}-OCCCI-V6.0-GLOBAL_MAPPED-PPD.nc")
        result_ds = xr.Dataset(
            {
                "VGPM": (["lat", "lon"], result_array)
            },
            coords={
                "lat": lat,
                "lon": lon
            },
            attrs={"generated_by": "Copilot process_one_date"}
        )
        result_ds.to_netcdf(out_file)

        print(f"✅ {date}: processed {np.sum(valid_mask)} pixels → {out_file}")

    except Exception as e:
        print(f"❌ {date}: failed with error → {e}")


def get_dates_from_filenames(prod,dataset=None):
    chl_files = get_prod_files("CHL",dataset=dataset)
    dates = []
    for f in chl_files:
        # Assumes filename like '20250701_something.nc' or '20250701.nc'
        basename = os.path.basename(f)
        match = re.search(r"(\d{8})", basename)
        if match:
            yyyymmdd = match.group(1)
            dates.append(yyyymmdd)
    return sorted(dates)

def normalize_dates(dates,format='yyyymmdd'):
    """
    Normalize flexible date inputs into YYYY-MM-DD strings.

    Input can be:
        dates (list): Accepts list of dates, range, year(s), or datetime objects.
            - List of date strings (yyyymmdd or yyyy-mm-dd)
            - List of datetime.date objects
            - List of integers: [2023] or [2023, 2025]
            - List of two strings: ['20250101', '20250630']
    
        format (str): Output format. Options:
            - 'yyyymmdd'       → default
            - 'yyyy-mm-dd'
            - 'yyyymmddhhmmss' → used if any input contains time info
            - 'datetime'       → returns datetime.date 
    Returns:
        list: Formatted date strings or date objects

    """
    def to_dt(d):
        if isinstance(d, int): return date(d, 1, 1)
        if isinstance(d, str):
            for fmt in ("%Y%m%d%H%M%S", "%Y%m%d", "%Y-%m-%d"):
                try:
                    return datetime.strptime(d, fmt)
                except ValueError:
                    continue
            raise ValueError(f"Could not parse string: {d}")
        if isinstance(d, (datetime, date)): return d
        raise TypeError(f"Unsupported date type: {type(d)}")

    if dates is None:
        return None

    # Range of years
    if all(isinstance(d, int) for d in dates):
        start = date(dates[0], 1, 1)
        end = date(dates[-1], 12, 31)

    # Range of strings
    elif len(dates) == 2 and all(isinstance(d, (str, int, date, datetime)) for d in dates):
        start = to_dt(dates[0])
        end = to_dt(dates[1])

    # Treat as list of individual dates
    else:
        return [format_date(to_dt(d), format) for d in dates]

    # Build range
    step = timedelta(days=1)
    current = start
    out = []
    while current <= end:
        out.append(format_date(current, format))
        current += step
    return out


def format_date(dt_obj, format):
    if format == 'datetime':
        return dt_obj.date() if isinstance(dt_obj, datetime) else dt_obj
    if format == 'yyyy-mm-dd':
        return dt_obj.strftime("%Y-%m-%d")
    if format == 'yyyymmddhhmmss':
        return dt_obj.strftime("%Y%m%d%H%M%S")
    return dt_obj.strftime("%Y%m%d")  # default


def build_pp_date_map(dates=None, get_date_prod="CHL", chl_dataset=None, sst_dataset=None, par_dataset=None):
    """
    Constructs a date→(CHL, SST, PAR) file path mapping using get_prod_files.

    Parameters:
        dates (list of str): List of dates (YYYYMMDD) to process.
        prod_chl, prod_sst, prod_par (str): Product identifiers for each input variable.

    Returns:
        dict: Mapping of date → (chl_path, sst_path, par_path)
    """
    
    def extract_date_from_path(path):
        match = re.search(r"(\d{8})", os.path.basename(path))
        return f"{match.group(1)[:4]}-{match.group(1)[4:6]}-{match.group(1)[6:]}" if match else None
    
    # Get files
    chl_files = get_prod_files("CHL", dataset=chl_dataset)
    sst_files = get_prod_files("SST", dataset=sst_dataset)
    par_files = get_prod_files("PAR", dataset=par_dataset)

    # Index by date
    chl_map = {extract_date_from_path(f): f for f in chl_files if extract_date_from_path(f)}
    sst_map = {extract_date_from_path(f): f for f in sst_files if extract_date_from_path(f)}
    par_map = {extract_date_from_path(f): f for f in par_files if extract_date_from_path(f)}

    # Determine which dates to use
    target_dates = normalize_dates(dates)
    if target_dates is None:
        target_dates = sorted(chl_map.keys())

     # Build mapping for dates present in CHL and all other inputs
    date_paths_map = {}
    for date in sorted(chl_map.keys()):
        if date in sst_map and date in par_map:
            date_paths_map[date] = (chl_map[date], sst_map[date], par_map[date])
        else:
            print(f"⚠ Missing data for {date}: CHL ✅  SST {'✅' if date in sst_map else '❌'}  PAR {'✅' if date in par_map else '❌'}")

    print(f"📅 Mapped {len(date_paths_map)} dates with complete product files.")
    return date_paths_map

def run_vgpm_batch(date_paths_map, output_dir, input_dirs):
    with ProcessPoolExecutor() as executor:
        futures = []
        for date, paths in date_paths_map.items():
            futures.append(executor.submit(
                process_one_date, date, paths, output_dir, input_dirs
            ))
        for f in futures:
            f.result()


In [21]:
dates = normalize_dates([2023,2024],format='datetime')
dates

[datetime.date(2023, 1, 1),
 datetime.date(2023, 1, 2),
 datetime.date(2023, 1, 3),
 datetime.date(2023, 1, 4),
 datetime.date(2023, 1, 5),
 datetime.date(2023, 1, 6),
 datetime.date(2023, 1, 7),
 datetime.date(2023, 1, 8),
 datetime.date(2023, 1, 9),
 datetime.date(2023, 1, 10),
 datetime.date(2023, 1, 11),
 datetime.date(2023, 1, 12),
 datetime.date(2023, 1, 13),
 datetime.date(2023, 1, 14),
 datetime.date(2023, 1, 15),
 datetime.date(2023, 1, 16),
 datetime.date(2023, 1, 17),
 datetime.date(2023, 1, 18),
 datetime.date(2023, 1, 19),
 datetime.date(2023, 1, 20),
 datetime.date(2023, 1, 21),
 datetime.date(2023, 1, 22),
 datetime.date(2023, 1, 23),
 datetime.date(2023, 1, 24),
 datetime.date(2023, 1, 25),
 datetime.date(2023, 1, 26),
 datetime.date(2023, 1, 27),
 datetime.date(2023, 1, 28),
 datetime.date(2023, 1, 29),
 datetime.date(2023, 1, 30),
 datetime.date(2023, 1, 31),
 datetime.date(2023, 2, 1),
 datetime.date(2023, 2, 2),
 datetime.date(2023, 2, 3),
 datetime.date(2023, 2, 4)

In [25]:
chl_path = get_prod_files('chl',dataset='OCCCI',getfilepath=True)
sst_path = get_prod_files('sst',dataset='CORALSST',getfilepath=True)
par_path = get_prod_files('par',dataset='GLOBCOLOUR',getfilepath=True)
input_dirs = [chl_path,sst_path,par_path]
input_dirs

['/Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/OCCCI/V6.0/SOURCE_DATA/MAPPED_4KM_DAILY/CHL',
 '/Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/CORALSST/V3.1/SOURCE_DATA/MAPPED_5KM_DAILY/SST',
 '/Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/GLOBCOLOUR/V4.2.1/SOURCE_DATA/MAPPED_4KM_DAILY/PAR']

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from functools import partial

def process_one_date(date, paths, lat, lon, output_dir, input_dirs):
    chl_path, sst_path, par_path = paths
    chl, sst, par = load_daily_inputs(date, chl_path, sst_path, par_path)
    daily_vgpm(date, chl, sst, par, lat, lon, output_dir, input_dirs)


In [ ]:
date_path_map = build_pp_date_map(sst_dataset='CORALSST')

chl_path = get_prod_files('chl',dataset='OCCCI',getfilepath=True)
sst_path = get_prod_files('sst',dataset='CORALSST',getfilepath=True)
par_path = get_prod_files('par',dataset='GLOBCOLOUR',getfilepath=True)
input_dirs = [chl_path,sst_path,par_path]

def run_vgpm_batch(date_paths_map, lat, lon, output_dir, input_dirs):
    with ProcessPoolExecutor() as executor:
        futures = []
        for date, paths in date_paths_map.items():
            futures.append(executor.submit(
                process_one_date, date, paths, lat, lon, output_dir, input_dirs
            ))
        for f in futures:
            f.result()  # Optional: catch exceptions


💡 Optional Enhancements
If you know all grids are regular, you can precompute and cache regrid weights using xESMF for even faster execution. This will precompute the interpolation mapping once, so it doesn’t get recalculated every time you regrid. This can save a ton of time if you’re regridding many variables or looping over many files.
Validate units and fill values before regridding to avoid cross-variable inconsistencies (e.g. 0.0 vs NaN vs masked arrays).
Add logging with simple markers like ✅ SST regridded to keep processing transparent.

🚀 How It Works
xESMF separates two steps:
Build the regridder: This creates a weight matrix based on source and target grids.
Apply the regridder: You reuse that weight matrix to quickly regrid each variable.

🧪 Example
import xesmf as xe

# Create regridder from SST grid → CHL grid
regridder = xe.Regridder(sst_ds, chl_ds, method="bilinear", reuse_weights=True)

# Apply to SST and PAR variables
sst = regridder(sst_ds["analysed_sst"])
par = regridder(par_ds["PAR_mean"])

reuse_weights=True tells xESMF to save/load the weights file in the background (in a .nc file).
You can control where it saves with filename="my_weights.nc" if you need portability.

🧠 Why It Matters for You
Batch Efficiency: If chl_ds is fixed across time, just create one regridder and reuse across all dates.
Parallel-Friendly: Each process/thread avoids recomputing weights independently.
Debuggability: You can inspect the generated weights file to understand interpolation behavior (e.g., land/sea masking).

In [ ]:
import xesmf as xe

# Create regridder from SST grid → CHL grid
regridder = xe.Regridder(sst_ds, chl_ds, method="bilinear", reuse_weights=True)

# ??? Does xe.Regridder "bilenear" return the same result as regrid.linear???

# Apply to SST and PAR variables
sst = regridder(sst_ds["analysed_sst"])
par = regridder(par_ds["PAR_mean"])


In [ ]:
# Load your dataset (must have lat/lon as 2D arrays if irregular)
dirdata = r'/Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/'
dirchl = os.path.join(dirdata,'OCCCI/V6.0/SOURCE_DATA/MAPPED_4KM_DAILY/CHL/')
ds = xr.open_mfdataset(os.path.join(dirchl,'*.nc'), combine='by_coords')

area_da = calc_mapped_pixel_area(ds['lat'].values, ds['lon'].values)